# BrandSphere AI — Week 7: Campaign Prediction (Machine Learning)
**CRS AI Capstone 2025–26**

Covers:
1. Feature engineering on 200,000 campaign records
2. Label encoding and scaling pipeline
3. Training Random Forest, Gradient Boosting, and Ridge regression
4. Cross-validation and model evaluation
5. Feature importance analysis
6. Saving models as .pkl for Streamlit deployment

> **Honest note:** The marketing dataset has synthetically generated outcome columns (ROI, Engagement, Conversion) with random values, resulting in R²≈0. This is documented transparently. The ML pipeline, feature engineering, and model training code is fully real and production-ready.

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn plotly 2>/dev/null
import subprocess, os, sys
r = subprocess.run(['git','clone','--quiet','https://github.com/iblamehemer/og','/content/BrandSphere'],capture_output=True,text=True)
os.chdir('/content/BrandSphere'); sys.path.insert(0,'/content/BrandSphere')
os.makedirs('assets/sample_exports',exist_ok=True)
print('✅ Ready')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'figure.facecolor':'#0C0D0F','axes.facecolor':'#141518','text.color':'white','axes.labelcolor':'white','xtick.color':'white','ytick.color':'white'})
import seaborn as sns
import pickle, json, warnings; warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
print('✅ Libraries loaded')

## 1. Load and Inspect Data

In [ ]:
df = pd.read_csv('datasets/raw/marketing_campaign_dataset.csv')
print(f'Shape: {df.shape}')
print(f'\nTarget variable statistics:')
for col in ['ROI','Engagement_Score','Conversion_Rate']:
    print(f'  {col:20s}: mean={df[col].mean():.3f}, std={df[col].std():.3f}, min={df[col].min():.3f}, max={df[col].max():.3f}')
df.head()

## 2. Feature Engineering

In [ ]:
# Clean and engineer features
df['Acquisition_Cost_Num'] = df['Acquisition_Cost'].str.replace('[$,]','',regex=True).astype(float)
df['Duration_Days']  = df['Duration'].str.extract(r'(\d+)').astype(float)
df['CTR']            = (df['Clicks'] / df['Impressions'] * 100).round(4)
df['Date']           = pd.to_datetime(df['Date'])
df['Month']          = df['Date'].dt.month
df['Quarter']        = df['Date'].dt.quarter
df['Cost_Per_Click'] = (df['Acquisition_Cost_Num'] / df['Clicks'].clip(lower=1)).round(2)
df['Impression_Rate']= (df['Impressions'] / df['Duration_Days'].clip(lower=1)).round(2)

# Label encode categorical columns
cat_cols = ['Campaign_Type','Channel_Used','Location','Language','Customer_Segment']
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col+'_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le
    print(f'  Encoded {col}: {le.classes_}')

print(f'\n✅ Feature engineering complete. Shape: {df.shape}')

## 3. Build Feature Matrix

In [ ]:
FEATURES = ['Campaign_Type_enc','Channel_Used_enc','Location_enc','Language_enc',
            'Customer_Segment_enc','Duration_Days','Acquisition_Cost_Num',
            'CTR','Clicks','Impressions','Month','Quarter','Cost_Per_Click','Impression_Rate']

TARGETS = {'ROI': df['ROI'], 'Engagement': df['Engagement_Score'], 'Conversion': df['Conversion_Rate']}

X = df[FEATURES].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature matrix shape: {X_scaled.shape}')
print(f'\nFeature statistics after scaling:')
scaled_df = pd.DataFrame(X_scaled, columns=FEATURES)
print(scaled_df.describe().round(3))

## 4. Train Models with Cross-Validation

In [ ]:
results = {}
trained_models = {}

for target_name, y in TARGETS.items():
    print(f'\n{"="*50}')
    print(f'TARGET: {target_name}')
    print(f'{"="*50}')
    
    X_tr, X_te, y_tr, y_te = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    
    models = {
        'RandomForest':    RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=20, random_state=42, n_jobs=-1),
        'GradientBoost':   GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42),
        'Ridge':           Ridge(alpha=10.0),
    }
    
    best_r2, best_model, best_name = -999, None, ""
    
    for name, m in models.items():
        m.fit(X_tr, y_tr)
        y_pred  = m.predict(X_te)
        r2      = r2_score(y_te, y_pred)
        rmse    = mean_squared_error(y_te, y_pred) ** 0.5
        mae     = mean_absolute_error(y_te, y_pred)
        
        # 5-fold cross-validation
        cv_scores = cross_val_score(m, X_scaled, y, cv=5, scoring='r2', n_jobs=-1)
        
        print(f'  {name:15s} | R²={r2:+.4f} | RMSE={rmse:.4f} | MAE={mae:.4f} | CV R²={cv_scores.mean():.4f}±{cv_scores.std():.4f}')
        
        if r2 > best_r2:
            best_r2, best_model, best_name = r2, m, name
    
    trained_models[target_name] = best_model
    results[target_name] = {'best_model': best_name, 'r2': round(best_r2,4), 'rmse': round(rmse,4)}
    print(f'  → Best: {best_name} (R²={best_r2:.4f})')

print(f'\n✅ All models trained')
print(f'\nNote: R²≈0 is expected — outcome columns in this dataset are synthetically random.')
print(f'The feature engineering, encoding, scaling and RF pipeline are fully functional.')

## 5. Feature Importance (Random Forest)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Random Forest Feature Importance by Target Variable', color='#C9A84C', fontsize=13)

for ax, (target_name, model) in zip(axes, trained_models.items()):
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        sorted_idx  = np.argsort(importances)[::-1]
        top_n = 10
        ax.barh(
            [FEATURES[i] for i in sorted_idx[:top_n]][::-1],
            importances[sorted_idx[:top_n]][::-1],
            color='#C9A84C', alpha=0.85
        )
        ax.set_title(f'{target_name} Prediction', color='white', fontsize=11)
        ax.set_xlabel('Importance', color='white')
    else:
        # Ridge: use coefficient magnitudes
        coefs = np.abs(model.coef_)
        sorted_idx = np.argsort(coefs)[::-1][:10]
        ax.barh(
            [FEATURES[i] for i in sorted_idx][::-1],
            coefs[sorted_idx][::-1],
            color='#3ECFB2', alpha=0.85
        )
        ax.set_title(f'{target_name} (Ridge Coefficients)', color='white', fontsize=11)

plt.tight_layout()
plt.savefig('assets/sample_exports/03_feature_importance.png', dpi=120, bbox_inches='tight', facecolor='#0C0D0F')
plt.show()
print('✅ Feature importance chart saved')

## 6. Save Models

In [ ]:
import os
os.makedirs('models', exist_ok=True)

fname_map = {'ROI':'roi_model','Engagement':'engagement_model','Conversion':'conversion_model'}
for target_name, model in trained_models.items():
    path = f'models/{fname_map[target_name]}.pkl'
    with open(path,'wb') as f:
        pickle.dump(model, f)
    size = os.path.getsize(path)
    print(f'  ✅ Saved {path} ({size/1024:.1f} KB) — {type(model).__name__}')

with open('models/encoders.pkl','wb') as f: pickle.dump(encoders, f)
with open('models/scaler.pkl','wb')   as f: pickle.dump(scaler, f)

# Save results JSON
with open('models/training_results.json','w') as f:
    json.dump(results, f, indent=2)

print(f'\n✅ All artifacts saved')
print(json.dumps(results, indent=2))

## 7. Prediction Demo

In [ ]:
# Test the saved models with a sample prediction
with open('models/roi_model.pkl','rb') as f:
    loaded_model = pickle.load(f)

# Build a sample feature vector
sample = pd.DataFrame([{
    'Campaign_Type_enc':    0,   # Email
    'Channel_Used_enc':     2,   # Instagram  
    'Location_enc':         3,   # New York
    'Language_enc':         0,   # English
    'Customer_Segment_enc': 1,   # Tech Enthusiasts
    'Duration_Days':        30,
    'Acquisition_Cost_Num': 5000,
    'CTR':                  3.2,
    'Clicks':               1500,
    'Impressions':          46875,
    'Month':                3,
    'Quarter':              1,
    'Cost_Per_Click':       3.33,
    'Impression_Rate':      1562.5,
}])

sample_scaled = scaler.transform(sample)
roi_pred = loaded_model.predict(sample_scaled)[0]
print(f'Sample prediction:')
print(f'  Input: Email campaign, Instagram, 30 days, $5,000 budget')
print(f'  Predicted ROI: {roi_pred:.2f}')
print(f'  (Note: prediction reflects dataset mean due to synthetic outcomes)')

## Summary

| Model | Target | R² | RMSE | Notes |
|---|---|---|---|---|
| Ridge | ROI | ≈0.0 | 1.74 | Dataset outcomes synthetic |
| Ridge | Engagement | ≈0.0 | 2.88 | Dataset outcomes synthetic |
| Ridge | Conversion | ≈0.0 | 0.04 | Dataset outcomes synthetic |

**Pipeline is production-ready.** The encoding, scaling, and inference code powers the live Campaign Analytics tab in BrandSphere AI.